# FS1 — Engineering feasibility research

Research-only hardware/dependency benchmark for Kaggle T4. This notebook does not clone or modify TRIAGE-EG, does not run TEAM-EVAL, does not open GT, and does not implement FS1. Heavy models run sequentially in disposable subprocesses; OCR uses an isolated virtual environment. Every candidate records exact revision, dependencies, RAM/VRAM, latency, offline reload, cleanup residual, and explicit verified-versus-estimated status.


In [ ]:
import os
import shutil
from pathlib import Path

DATA_INPUT = Path(os.environ.get("AIC_DATA_ROOT", "/kaggle/input/datasets/nadkli/dataset-aic"))
OUTPUT_ROOT = Path("/kaggle/working/fs1_engineering_feasibility")
WORK_ROOT = Path("/kaggle/working/fs1_engineering_work")
ZIP_PATH = Path("/kaggle/working/fs1_engineering_feasibility_bundle.zip")
REPEATS = int(os.environ.get("AIC_FS1_REPEATS", "5"))
SAMPLE_VIDEO_COUNT = int(os.environ.get("AIC_FS1_SAMPLE_VIDEO_COUNT", "5"))
DELETE_MODEL_CACHE_AFTER_EACH = os.environ.get("AIC_FS1_KEEP_MODEL_CACHE", "0") != "1"

RUN_XCLIP = True
RUN_INTERNVIDEO_DISCOVERY = True
RUN_WHISPER_TURBO = True
RUN_QWEN25VL3B_FP16 = True
RUN_QWEN25VL3B_AWQ = True
RUN_SMOLVLM2_500M = True
RUN_GROUNDING_DINO_TINY = True
RUN_SAM21_TINY = True
RUN_PPOCRV5 = True
RUN_CLAP = False

MODEL_IDS = {
    "xclip": "microsoft/xclip-base-patch32",
    "whisper_turbo": "openai/whisper-large-v3-turbo",
    "whisper_local_fallback": "openai/whisper-small",
    "qwen_fp16": "Qwen/Qwen2.5-VL-3B-Instruct",
    "qwen_awq": "Qwen/Qwen2.5-VL-3B-Instruct-AWQ",
    "smolvlm": "HuggingFaceTB/SmolVLM2-500M-Video-Instruct",
    "grounding_dino": "IDEA-Research/grounding-dino-tiny",
    "sam2": "facebook/sam2.1-hiera-tiny",
}

for path in (OUTPUT_ROOT, WORK_ROOT):
    if path.exists():
        if Path("/kaggle/working") not in path.parents:
            raise RuntimeError(f"Refusing cleanup outside /kaggle/working: {path}")
        shutil.rmtree(path)
    path.mkdir(parents=True)
ZIP_PATH.unlink(missing_ok=True)
print({
    "task": "FS1_ENGINEERING_FEASIBILITY_RESEARCH_ONLY",
    "required_input": {"raw_aic_dataset": str(DATA_INPUT)},
    "forbidden_inputs": ["TEAM_EVAL", "GT", "BCF_WORKING_TREE"],
    "internet_required": "YES_FOR_ONE_TIME_RESEARCH_DOWNLOADS",
    "offline_reload_tested_per_model": True,
    "output_zip": str(ZIP_PATH),
    "models_execute_sequentially": True,
    "siglip2_rebuilt": False,
})


In [ ]:
import importlib.metadata
import json
import platform
import subprocess
import sys
import time

import psutil
import torch

def command_output(command):
    result = subprocess.run(command, capture_output=True, text=True, check=False)
    return {
        "command": command,
        "returncode": result.returncode,
        "stdout": result.stdout.strip(),
        "stderr": result.stderr.strip(),
    }

def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

gpu = None
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    gpu = {
        "name": props.name,
        "compute_capability": f"{props.major}.{props.minor}",
        "total_vram_bytes": props.total_memory,
        "bf16_supported_by_torch": bool(torch.cuda.is_bf16_supported()),
    }
SYSTEM_PROFILE = {
    "captured_at_epoch": time.time(),
    "python": sys.version,
    "platform": platform.platform(),
    "gpu": gpu,
    "cuda_runtime": torch.version.cuda,
    "torch": torch.__version__,
    "torchvision": package_version("torchvision"),
    "transformers": package_version("transformers"),
    "system_ram_total_bytes": psutil.virtual_memory().total,
    "system_ram_available_bytes": psutil.virtual_memory().available,
    "working_disk": shutil.disk_usage("/kaggle/working")._asdict(),
    "nvidia_smi": command_output(["nvidia-smi"]),
    "nvcc": command_output(["nvcc", "--version"]),
    "ffmpeg": command_output(["ffmpeg", "-version"]),
    "ffprobe": command_output(["ffprobe", "-version"]),
    "flash_attention_2_assumed": False,
    "bf16_assumed": False,
}
if not torch.cuda.is_available():
    raise RuntimeError("FS1 T4 research requires a CUDA accelerator")
if "T4" not in SYSTEM_PROFILE["gpu"]["name"]:
    SYSTEM_PROFILE["hardware_profile_warning"] = "RUN_IS_NOT_ON_NVIDIA_T4"
(OUTPUT_ROOT / "fs1_kaggle_runtime_profile.json").write_text(
    json.dumps(SYSTEM_PROFILE, indent=2) + "\n", encoding="utf-8"
)
print(json.dumps(SYSTEM_PROFILE, indent=2))


In [ ]:
import hashlib
from collections import defaultdict

def bounded_video_inventory(root, limit=10000):
    videos = []
    for path in sorted(Path(root).rglob("*.mp4")):
        videos.append(path)
        if len(videos) >= limit:
            break
    return videos

def ffprobe(path):
    result = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration:stream=index,codec_type,codec_name,width,height,r_frame_rate", "-of", "json", str(path)],
        capture_output=True, text=True, check=False,
    )
    if result.returncode:
        return {"status": "FAIL", "stderr": result.stderr.strip()}
    return {"status": "PASS", **json.loads(result.stdout)}

videos = bounded_video_inventory(DATA_INPUT)
if not videos:
    raise RuntimeError(f"No MP4 samples discovered below {DATA_INPUT}")
by_partition = defaultdict(list)
for path in videos:
    partition = next((part for part in path.parts if part.startswith("Videos_")), "UNKNOWN")
    by_partition[partition].append(path)
selected = []
for partition in sorted(by_partition):
    selected.append(by_partition[partition][0])
    if len(selected) >= SAMPLE_VIDEO_COUNT:
        break
if len(selected) < SAMPLE_VIDEO_COUNT:
    selected.extend(path for path in videos if path not in selected)
selected = selected[:SAMPLE_VIDEO_COUNT]

SAMPLE_ROOT = WORK_ROOT / "samples"
SAMPLE_ROOT.mkdir(parents=True, exist_ok=True)
sample_rows, all_frames, audio_path = [], [], None
for index, video in enumerate(selected):
    metadata = ffprobe(video)
    duration = float(metadata.get("format", {}).get("duration", 0.0) or 0.0)
    frame_paths = []
    fractions = tuple((step + 1) / 9 for step in range(8)) if index == 0 else (0.1, 0.5, 0.9)
    for frame_index, fraction in enumerate(fractions):
        frame_path = SAMPLE_ROOT / f"video_{index:02d}_frame_{frame_index:02d}.jpg"
        timestamp = max(0.0, duration * fraction)
        process = subprocess.run(
            ["ffmpeg", "-hide_banner", "-loglevel", "error", "-ss", str(timestamp), "-i", str(video), "-frames:v", "1", "-q:v", "2", "-y", str(frame_path)],
            capture_output=True, text=True, check=False,
        )
        if process.returncode == 0 and frame_path.is_file():
            frame_paths.append(frame_path)
            all_frames.append(frame_path)
    has_audio = any(stream.get("codec_type") == "audio" for stream in metadata.get("streams", []))
    if has_audio and audio_path is None:
        candidate = SAMPLE_ROOT / "audio_sample_30s.wav"
        process = subprocess.run(
            ["ffmpeg", "-hide_banner", "-loglevel", "error", "-i", str(video), "-t", "30", "-ac", "1", "-ar", "16000", "-y", str(candidate)],
            capture_output=True, text=True, check=False,
        )
        if process.returncode == 0 and candidate.is_file():
            audio_path = candidate
    sample_rows.append({
        "sample_id": f"FS1_VIDEO_{index:02d}",
        "video_path": str(video),
        "video_sha256_prefix": hashlib.sha256(video.read_bytes()[:1024 * 1024]).hexdigest(),
        "partition": next((part for part in video.parts if part.startswith("Videos_")), "UNKNOWN"),
        "duration_seconds": duration,
        "has_audio": has_audio,
        "frames": [str(path) for path in frame_paths],
        "ffprobe": metadata,
    })
if not all_frames:
    raise RuntimeError("Frame extraction failed for every bounded sample")
SAMPLE_MANIFEST = {
    "selection_policy": "FIRST_VIDEO_PER_SORTED_DISTINCT_PARTITION_THEN_SORTED_FILL",
    "video_inventory_count_bounded": len(videos),
    "inventory_limit": 10000,
    "samples": sample_rows,
    "audio_sample": str(audio_path) if audio_path else None,
    "required_scene_categories": {
        "static_scene": "REQUIRES_POST_RUN_VISUAL_REVIEW",
        "action_transition": "MULTI_TIMESTAMP_SAMPLE_PROXY",
        "readable_text": "OCR_RESULT_WILL_CONFIRM_OR_REJECT",
        "audio": bool(audio_path),
    },
}
(OUTPUT_ROOT / "fs1_sample_manifest.json").write_text(
    json.dumps(SAMPLE_MANIFEST, indent=2) + "\n", encoding="utf-8"
)
print(json.dumps(SAMPLE_MANIFEST, indent=2))


In [ ]:
INSTALL_COMMAND = [
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    "transformers>=4.57.0,<5", "accelerate>=1.10,<2",
    "qwen-vl-utils>=0.0.14,<0.1", "av>=14,<17", "soundfile>=0.13,<1",
]
install_started = time.perf_counter()
install_process = subprocess.run(INSTALL_COMMAND, capture_output=True, text=True, check=False)
INSTALL_SECONDS = time.perf_counter() - install_started
INSTALL_RESULT = {
    "command": INSTALL_COMMAND,
    "seconds": INSTALL_SECONDS,
    "returncode": install_process.returncode,
    "stdout_tail": install_process.stdout.splitlines()[-40:],
    "stderr_tail": install_process.stderr.splitlines()[-40:],
}
if install_process.returncode:
    raise RuntimeError("Bounded FS1 research dependencies failed to install: " + install_process.stderr[-2000:])

WORKER_SOURCE = r'''import gc, hashlib, json, os, statistics, sys, time, traceback
from pathlib import Path
import psutil

request = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
output_path = Path(sys.argv[2])
result = {
    "candidate": request["candidate"], "model_id": request["model_id"],
    "status": "FAIL", "verified_on": "KAGGLE_RUNTIME", "errors": [],
}

def rss():
    return psutil.Process().memory_info().rss

def move(value, device):
    if hasattr(value, "to"):
        return value.to(device)
    if isinstance(value, dict):
        return {key: move(item, device) for key, item in value.items()}
    return value

def shape(value):
    if hasattr(value, "shape"):
        return list(value.shape)
    if isinstance(value, dict):
        return {key: shape(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [shape(item) for item in value[:5]]
    return type(value).__name__

try:
    import_start = time.perf_counter()
    import torch
    from PIL import Image
    from huggingface_hub import HfApi, snapshot_download
    import_seconds = time.perf_counter() - import_start
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    info = HfApi().model_info(request["model_id"])
    revision = info.sha
    result.update({
        "exact_revision": revision,
        "license": (info.card_data or {}).get("license") if info.card_data else None,
        "import_seconds": import_seconds,
        "dependencies": request.get("dependencies", []),
    })
    snapshot = Path(snapshot_download(
        request["model_id"], revision=revision,
        cache_dir=request["cache_dir"], local_dir=request["snapshot_dir"],
        ignore_patterns=["*.h5", "*.msgpack", "*.onnx", "pytorch_model.bin"],
    ))
    inventory = []
    for path in sorted(item for item in snapshot.rglob("*") if item.is_file()):
        digest = hashlib.sha256()
        with path.open("rb") as handle:
            for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
                digest.update(block)
        inventory.append({
            "path": path.relative_to(snapshot).as_posix(),
            "size_bytes": path.stat().st_size,
            "sha256": digest.hexdigest(),
        })
    result["file_inventory"] = inventory
    result["asset_size_bytes"] = sum(item["size_bytes"] for item in inventory)
    result["snapshot_path"] = str(snapshot)
    frames = [Image.open(path).convert("RGB") for path in request["frames"]]
    first_image = frames[0]
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    ram_before = rss()
    load_start = time.perf_counter()
    candidate = request["candidate"]
    processor = model = None

    if candidate == "XCLIP":
        from transformers import XCLIPModel, XCLIPProcessor
        processor = XCLIPProcessor.from_pretrained(snapshot, local_files_only=True)
        model = XCLIPModel.from_pretrained(snapshot, local_files_only=True, torch_dtype=dtype).to(device).eval()
        inputs = processor(text=["a person is moving"], videos=frames[:8], return_tensors="pt", padding=True)
        inputs = move(inputs, device)
        infer = lambda: model(**inputs).logits_per_video
        reload_class = XCLIPModel
    elif candidate == "WHISPER_TURBO":
        import soundfile as sf
        from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor
        processor = AutoProcessor.from_pretrained(snapshot, local_files_only=True)
        model = AutoModelForSpeechSeq2Seq.from_pretrained(snapshot, local_files_only=True, torch_dtype=dtype).to(device).eval()
        if request.get("audio"):
            waveform, sampling_rate = sf.read(request["audio"], dtype="float32")
        else:
            import numpy as np
            sampling_rate, waveform = 16000, np.zeros(16000 * 10, dtype="float32")
        inputs = processor(waveform, sampling_rate=sampling_rate, return_tensors="pt")
        features = inputs.input_features.to(device, dtype=dtype)
        infer = lambda: model.generate(features, max_new_tokens=32)
        reload_class = AutoModelForSpeechSeq2Seq
        result["audio_seconds"] = len(waveform) / sampling_rate
        result["timestamp_artifact_contract"] = "segment_start_end_text_jsonl"
    elif candidate in {"QWEN25VL3B_FP16", "QWEN25VL3B_AWQ"}:
        from qwen_vl_utils import process_vision_info
        from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
        processor = AutoProcessor.from_pretrained(
            snapshot, local_files_only=True, min_pixels=256 * 28 * 28, max_pixels=512 * 28 * 28
        )
        kwargs = {"local_files_only": True, "device_map": "auto", "attn_implementation": "sdpa"}
        if candidate.endswith("FP16"):
            kwargs["torch_dtype"] = torch.float16
        model = Qwen2_5_VLForConditionalGeneration.from_pretrained(snapshot, **kwargs).eval()
        messages = [{"role": "user", "content": [{"type": "image", "image": request["frames"][0]}, {"type": "text", "text": "Describe the visible evidence in one short sentence."}]}]
        prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(text=[prompt], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt")
        inputs = move(inputs, device)
        infer = lambda: model.generate(**inputs, max_new_tokens=24)
        reload_class = Qwen2_5_VLForConditionalGeneration
        result["visual_token_bounds"] = {"min_pixels": 256 * 28 * 28, "max_pixels": 512 * 28 * 28}
        result["attention_implementation"] = "sdpa_no_flash_attention_2"
    elif candidate == "SMOLVLM2_500M":
        from transformers import AutoModelForImageTextToText, AutoProcessor
        processor = AutoProcessor.from_pretrained(snapshot, local_files_only=True)
        model = AutoModelForImageTextToText.from_pretrained(snapshot, local_files_only=True, torch_dtype=dtype).to(device).eval()
        messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": "Describe this frame briefly."}]}]
        prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
        inputs = processor(text=prompt, images=[first_image], return_tensors="pt")
        inputs = move(inputs, device)
        infer = lambda: model.generate(**inputs, max_new_tokens=24)
        reload_class = AutoModelForImageTextToText
        result["video_fallback_probe"] = "SINGLE_FRAME_RESOURCE_FLOOR; VIDEO_API_REQUIRES_SEPARATE_VALIDATION"
    elif candidate == "GROUNDING_DINO_TINY":
        from transformers import AutoModelForZeroShotObjectDetection, AutoProcessor
        processor = AutoProcessor.from_pretrained(snapshot, local_files_only=True)
        model = AutoModelForZeroShotObjectDetection.from_pretrained(snapshot, local_files_only=True, torch_dtype=dtype).to(device).eval()
        inputs = processor(images=first_image, text="person. car. sign.", return_tensors="pt")
        inputs = move(inputs, device)
        infer = lambda: model(**inputs)
        reload_class = AutoModelForZeroShotObjectDetection
    elif candidate == "SAM21_TINY":
        from transformers import Sam2Model, Sam2Processor
        processor = Sam2Processor.from_pretrained(snapshot, local_files_only=True)
        model = Sam2Model.from_pretrained(snapshot, local_files_only=True, torch_dtype=dtype).to(device).eval()
        width, height = first_image.size
        inputs = processor(images=first_image, input_points=[[[width / 2, height / 2]]], input_labels=[[[1]]], return_tensors="pt")
        inputs = move(inputs, device)
        infer = lambda: model(**inputs)
        reload_class = Sam2Model
        result["optional_cuda_extension_required"] = False
    else:
        raise ValueError(f"Unknown worker candidate: {candidate}")

    result["load_seconds"] = time.perf_counter() - load_start
    result["cpu_ram_before_load_bytes"] = ram_before
    result["cpu_ram_after_load_bytes"] = rss()
    result["gpu_after_load_allocated_bytes"] = torch.cuda.memory_allocated()
    result["gpu_after_load_reserved_bytes"] = torch.cuda.memory_reserved()
    with torch.inference_mode():
        warm_start = time.perf_counter(); warm_output = infer(); torch.cuda.synchronize()
        result["warmup_seconds"] = time.perf_counter() - warm_start
        latencies = []
        for _ in range(request["repeats"]):
            start = time.perf_counter(); measured = infer(); torch.cuda.synchronize()
            latencies.append(time.perf_counter() - start)
    result["latency_seconds"] = {
        "repeats": len(latencies), "values": latencies,
        "p50": statistics.median(latencies),
        "p95": sorted(latencies)[max(0, int(0.95 * len(latencies)) - 1)],
    }
    result["output_shape"] = shape(measured)
    result["finite_output"] = True
    for value in ([measured] if hasattr(measured, "shape") else []):
        result["finite_output"] = bool(torch.isfinite(value.float()).all().item())
    result["gpu_peak_allocated_bytes"] = torch.cuda.max_memory_allocated()
    result["gpu_peak_reserved_bytes"] = torch.cuda.max_memory_reserved()
    del measured, warm_output, inputs, model, processor
    gc.collect(); torch.cuda.empty_cache()
    reload_start = time.perf_counter()
    processor2 = None
    if candidate == "SAM21_TINY":
        from transformers import Sam2Processor
        processor2 = Sam2Processor.from_pretrained(snapshot, local_files_only=True)
    else:
        from transformers import AutoProcessor
        processor2 = AutoProcessor.from_pretrained(snapshot, local_files_only=True)
    reload_kwargs = {"local_files_only": True}
    if candidate not in {"QWEN25VL3B_AWQ"}:
        reload_kwargs["torch_dtype"] = dtype
    model2 = reload_class.from_pretrained(snapshot, **reload_kwargs)
    result["offline_local_files_only_reload"] = "PASS"
    result["offline_reload_seconds"] = time.perf_counter() - reload_start
    del model2, processor2, frames, first_image
    gc.collect(); torch.cuda.empty_cache()
    result["gpu_after_unload_allocated_bytes"] = torch.cuda.memory_allocated()
    result["gpu_after_unload_reserved_bytes"] = torch.cuda.memory_reserved()
    result["cpu_ram_after_unload_bytes"] = rss()
    peak = result["gpu_peak_reserved_bytes"]
    total = torch.cuda.get_device_properties(0).total_memory
    result["status"] = "PASS_WITH_LIMITS" if peak > total * 0.85 else "PASS"
    if peak <= 3.2 * 1024**3:
        result["rtx3050ti_4gb_status"] = "ESTIMATED_FEASIBLE"
    elif peak <= 5.0 * 1024**3:
        result["rtx3050ti_4gb_status"] = "ESTIMATED_WITH_CPU_OFFLOAD"
    else:
        result["rtx3050ti_4gb_status"] = "ESTIMATED_NOT_FEASIBLE"
except Exception as error:
    result["errors"].append({"type": type(error).__name__, "message": str(error), "traceback": traceback.format_exc()[-8000:]})
    result.setdefault("offline_local_files_only_reload", "FAIL")
    result.setdefault("rtx3050ti_4gb_status", "UNKNOWN")
finally:
    try:
        import torch
        gc.collect(); torch.cuda.empty_cache()
        result["final_gpu_allocated_bytes"] = torch.cuda.memory_allocated() if torch.cuda.is_available() else 0
        result["final_gpu_reserved_bytes"] = torch.cuda.memory_reserved() if torch.cuda.is_available() else 0
    except Exception:
        pass
    output_path.write_text(json.dumps(result, indent=2, default=str) + "\n", encoding="utf-8")
'''

WORKER_PATH = WORK_ROOT / "fs1_model_worker.py"
WORKER_PATH.write_text(WORKER_SOURCE, encoding="utf-8")
compile(WORKER_SOURCE, str(WORKER_PATH), "exec")

RESULTS = []
COMMAND_LOG = [INSTALL_RESULT]

def run_model_candidate(candidate, model_id, dependencies):
    slug = candidate.casefold()
    candidate_root = WORK_ROOT / "model_runs" / slug
    candidate_root.mkdir(parents=True, exist_ok=True)
    request = {
        "candidate": candidate,
        "model_id": model_id,
        "dependencies": dependencies,
        "cache_dir": str(candidate_root / "hf_cache"),
        "snapshot_dir": str(candidate_root / "snapshot"),
        "frames": [str(path) for path in all_frames[:8]],
        "audio": str(audio_path) if audio_path else None,
        "repeats": REPEATS,
    }
    request_path = candidate_root / "request.json"
    result_path = OUTPUT_ROOT / f"candidate_{slug}.json"
    request_path.write_text(json.dumps(request, indent=2) + "\n", encoding="utf-8")
    command = [sys.executable, str(WORKER_PATH), str(request_path), str(result_path)]
    started = time.perf_counter()
    process = subprocess.run(command, capture_output=True, text=True, check=False)
    elapsed = time.perf_counter() - started
    COMMAND_LOG.append({
        "candidate": candidate, "command": command, "seconds": elapsed,
        "returncode": process.returncode,
        "stdout_tail": process.stdout.splitlines()[-30:],
        "stderr_tail": process.stderr.splitlines()[-50:],
    })
    if result_path.is_file():
        value = json.loads(result_path.read_text(encoding="utf-8"))
    else:
        value = {"candidate": candidate, "model_id": model_id, "status": "FAIL", "errors": [{"message": "worker did not create result"}]}
    value["subprocess_wall_seconds"] = elapsed
    RESULTS.append(value)
    if DELETE_MODEL_CACHE_AFTER_EACH and candidate_root.exists():
        shutil.rmtree(candidate_root)
    print(json.dumps(value, indent=2))
    return value

print({"dependency_install": INSTALL_RESULT, "worker_ast": "PASS"})


In [ ]:
from huggingface_hub import HfApi

if RUN_XCLIP:
    run_model_candidate(
        "XCLIP", MODEL_IDS["xclip"],
        ["torch", "torchvision", "transformers", "accelerate", "Pillow"],
    )

if RUN_INTERNVIDEO_DISCOVERY:
    started = time.perf_counter()
    try:
        matches = [
            {"id": item.id, "sha": item.sha, "library_name": item.library_name,
             "pipeline_tag": item.pipeline_tag, "private": item.private, "gated": item.gated}
            for item in HfApi().list_models(author="OpenGVLab", search="InternVideo2", limit=100)
        ]
        transformers_distilled = [
            row for row in matches
            if row.get("library_name") == "transformers"
            and any(token in row["id"].casefold() for token in ("s14", "s-14", "b14", "b-14", "distill"))
        ]
        status = "PASS" if transformers_distilled else "FAIL"
        reason = None if transformers_distilled else "NO_NON_GATED_OFFICIAL_TRANSFORMERS_DISTILLED_S14_OR_B14_FOUND"
        result = {
            "candidate": "INTERNVIDEO2_DISTILLED_DISCOVERY",
            "model_id": None,
            "status": status,
            "discovery_seconds": time.perf_counter() - started,
            "official_author_matches": matches,
            "eligible_transformers_distilled_matches": transformers_distilled,
            "blocker": reason,
            "dependency_risk": True,
            "rtx3050ti_4gb_status": "UNKNOWN",
            "inference_executed": False,
        }
    except Exception as error:
        result = {
            "candidate": "INTERNVIDEO2_DISTILLED_DISCOVERY", "status": "FAIL",
            "errors": [{"type": type(error).__name__, "message": str(error)}],
            "dependency_risk": True, "inference_executed": False,
            "rtx3050ti_4gb_status": "UNKNOWN",
        }
    RESULTS.append(result)
    print(json.dumps(result, indent=2))


In [ ]:
if RUN_WHISPER_TURBO:
    run_model_candidate(
        "WHISPER_TURBO", MODEL_IDS["whisper_turbo"],
        ["torch", "transformers", "accelerate", "soundfile", "ffmpeg"],
    )

if RUN_QWEN25VL3B_FP16:
    run_model_candidate(
        "QWEN25VL3B_FP16", MODEL_IDS["qwen_fp16"],
        ["torch", "torchvision", "transformers", "accelerate", "qwen-vl-utils", "Pillow"],
    )
if RUN_QWEN25VL3B_AWQ:
    run_model_candidate(
        "QWEN25VL3B_AWQ", MODEL_IDS["qwen_awq"],
        ["torch", "torchvision", "transformers", "accelerate", "qwen-vl-utils", "autoawq/awq-runtime-if-required"],
    )
if RUN_SMOLVLM2_500M:
    run_model_candidate(
        "SMOLVLM2_500M", MODEL_IDS["smolvlm"],
        ["torch", "torchvision", "transformers", "accelerate", "Pillow"],
    )


In [ ]:
if RUN_GROUNDING_DINO_TINY:
    run_model_candidate(
        "GROUNDING_DINO_TINY", MODEL_IDS["grounding_dino"],
        ["torch", "torchvision", "transformers", "accelerate", "Pillow"],
    )
if RUN_SAM21_TINY:
    run_model_candidate(
        "SAM21_TINY", MODEL_IDS["sam2"],
        ["torch", "torchvision", "transformers", "accelerate", "Pillow"],
    )
if not RUN_CLAP:
    RESULTS.append({
        "candidate": "CLAP", "status": "NOT_RUN_OPTIONAL",
        "reason": "OPTIONAL_AND_MUST_NOT_BLOCK_FS1", "rtx3050ti_4gb_status": "UNKNOWN",
    })


In [ ]:
OCR_RESULT_PATH = OUTPUT_ROOT / "candidate_ppocrv5.json"
if RUN_PPOCRV5:
    ocr_root = WORK_ROOT / "ocr_isolated"
    venv_root = ocr_root / "venv"
    ocr_root.mkdir(parents=True, exist_ok=True)
    create_started = time.perf_counter()
    create_process = subprocess.run([sys.executable, "-m", "venv", str(venv_root)], capture_output=True, text=True, check=False)
    venv_python = venv_root / "bin/python"
    install_command = [
        str(venv_python), "-m", "pip", "install", "--disable-pip-version-check",
        "paddleocr==3.7.0", "paddlepaddle-gpu==3.2.0", "psutil>=6,<8",
    ]
    install_process = subprocess.run(install_command, capture_output=True, text=True, check=False) if create_process.returncode == 0 else create_process
    install_seconds = time.perf_counter() - create_started
    ocr_worker = ocr_root / "ocr_worker.py"
    ocr_worker.write_text(r'''import json, sys, time, traceback
from pathlib import Path
import psutil
out = {"candidate": "PPOCRV5_MOBILE_VI", "status": "FAIL", "rtx3050ti_4gb_status": "UNKNOWN"}
try:
    start = time.perf_counter()
    import paddle
    from paddleocr import PaddleOCR
    out["import_seconds"] = time.perf_counter() - start
    out["paddle"] = paddle.__version__
    load = time.perf_counter()
    engine = PaddleOCR(
        lang="vi", ocr_version="PP-OCRv5",
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
        device="gpu:0",
    )
    out["load_seconds"] = time.perf_counter() - load
    latencies, outputs = [], []
    for image in sys.argv[2:]:
        started = time.perf_counter(); prediction = list(engine.predict(image)); latencies.append(time.perf_counter() - started)
        outputs.append({"image": image, "result_count": len(prediction), "repr": repr(prediction)[:2000]})
    out.update({
        "status": "PASS", "latency_seconds": {"values": latencies, "p50": sorted(latencies)[len(latencies)//2]},
        "outputs": outputs, "cpu_ram_after_bytes": psutil.Process().memory_info().rss,
        "offline_asset_materialization": "MODEL_CACHE_CREATED_BY_PADDLEOCR; HASH_INVENTORY_REQUIRED_FOR_FINAL_ASSET",
        "rtx3050ti_4gb_status": "ESTIMATED_FEASIBLE",
    })
except Exception as error:
    out["errors"] = [{"type": type(error).__name__, "message": str(error), "traceback": traceback.format_exc()[-8000:]}]
Path(sys.argv[1]).write_text(json.dumps(out, indent=2, default=str) + "\n", encoding="utf-8")
''', encoding="utf-8")
    run_command = [str(venv_python), str(ocr_worker), str(OCR_RESULT_PATH), *[str(path) for path in all_frames[:5]]]
    run_started = time.perf_counter()
    run_process = subprocess.run(run_command, capture_output=True, text=True, check=False) if install_process.returncode == 0 else install_process
    run_seconds = time.perf_counter() - run_started
    if OCR_RESULT_PATH.is_file():
        ocr_result = json.loads(OCR_RESULT_PATH.read_text(encoding="utf-8"))
    else:
        ocr_result = {"candidate": "PPOCRV5_MOBILE_VI", "status": "FAIL", "errors": [{"message": "install or worker failed"}], "rtx3050ti_4gb_status": "UNKNOWN"}
    ocr_result.update({
        "isolated_venv": True, "install_command": install_command,
        "install_seconds": install_seconds, "install_returncode": install_process.returncode,
        "install_stderr_tail": install_process.stderr.splitlines()[-60:],
        "worker_command": run_command, "worker_seconds": run_seconds,
        "worker_returncode": run_process.returncode,
    })
    OCR_RESULT_PATH.write_text(json.dumps(ocr_result, indent=2) + "\n", encoding="utf-8")
    RESULTS.append(ocr_result)
    COMMAND_LOG.extend([
        {"candidate": "PPOCRV5_MOBILE_VI", "command": install_command, "seconds": install_seconds, "returncode": install_process.returncode},
        {"candidate": "PPOCRV5_MOBILE_VI", "command": run_command, "seconds": run_seconds, "returncode": run_process.returncode},
    ])
    if DELETE_MODEL_CACHE_AFTER_EACH and ocr_root.exists():
        shutil.rmtree(ocr_root)
    print(json.dumps(ocr_result, indent=2))


In [ ]:
import math

result_map = {row["candidate"]: row for row in RESULTS}
def passed(name):
    return result_map.get(name, {}).get("status") in {"PASS", "PASS_WITH_LIMITS"}

video_recommendation = "XCLIP" if passed("XCLIP") else "NO_VIDEO_ENCODER"
if passed("QWEN25VL3B_FP16") and passed("SMOLVLM2_500M"):
    vlm_recommendation = "DUAL_PROFILE"
elif passed("QWEN25VL3B_FP16"):
    vlm_recommendation = "QWEN25VL3B"
elif passed("SMOLVLM2_500M"):
    vlm_recommendation = "SMOLVLM2_500M"
else:
    vlm_recommendation = "NO_VLM"

CLASSIFICATIONS = {
    "frozen_clip_siglip2_opus_btc_t3_g1_bcf": "FS1_CORE",
    "xclip": "FS1_CORE" if passed("XCLIP") else "DROP_FOR_PRELIM",
    "internvideo2_distilled": "DEFER",
    "ppocrv5": "PRECOMPUTE_ONLY" if passed("PPOCRV5_MOBILE_VI") else "DEFER",
    "whisper_turbo": "PRECOMPUTE_ONLY" if passed("WHISPER_TURBO") else "DEFER",
    "whisper_small": "LOCAL_FALLBACK_ONLY",
    "qwen25vl3b": "FS1_OPTIONAL_PLUGIN" if passed("QWEN25VL3B_FP16") else "DROP_FOR_PRELIM",
    "qwen25vl3b_awq": "FS1_OPTIONAL_PLUGIN" if passed("QWEN25VL3B_AWQ") else "DEFER",
    "smolvlm2_500m": "LOCAL_FALLBACK_ONLY" if passed("SMOLVLM2_500M") else "DEFER",
    "grounding_dino_tiny": "FS1_OPTIONAL_PLUGIN" if passed("GROUNDING_DINO_TINY") else "DROP_FOR_PRELIM",
    "sam21_tiny": "FS1_OPTIONAL_PLUGIN" if passed("SAM21_TINY") else "DROP_FOR_PRELIM",
    "clap": "DEFER",
    "query_local_event_graph": "FS1_CORE",
}

sample_durations = [row["duration_seconds"] for row in SAMPLE_MANIFEST["samples"] if row["duration_seconds"] > 0]
average_video_seconds = sum(sample_durations) / len(sample_durations) if sample_durations else None
EXTRAPOLATIONS = {
    "labels": "ESTIMATED_FROM_BOUNDED_SAMPLE_NOT_FULL_CORPUS_MEASUREMENT",
    "btc_keyframes": 177321,
    "video_inventory_count_bounded": SAMPLE_MANIFEST["video_inventory_count_bounded"],
    "average_sample_video_seconds": average_video_seconds,
}
ocr = result_map.get("PPOCRV5_MOBILE_VI", {})
ocr_latencies = ocr.get("latency_seconds", {}).get("values", [])
if ocr_latencies:
    EXTRAPOLATIONS["ocr"] = {
        "seconds": 177321 * (sum(ocr_latencies) / len(ocr_latencies)),
        "hours": 177321 * (sum(ocr_latencies) / len(ocr_latencies)) / 3600,
        "storage_assumption": "JSONL text boxes/confidence only; measure output bytes before master task",
    }
whisper = result_map.get("WHISPER_TURBO", {})
if whisper.get("audio_seconds") and whisper.get("latency_seconds", {}).get("p50"):
    rtf = whisper["latency_seconds"]["p50"] / whisper["audio_seconds"]
    EXTRAPOLATIONS["asr"] = {
        "measured_realtime_factor": rtf,
        "corpus_audio_duration_unknown": True,
        "estimated_hours_if_every_bounded_video_has_average_duration": (
            len(videos) * average_video_seconds * rtf / 3600 if average_video_seconds else None
        ),
        "artifact": "timestamped transcript JSONL plus lightweight text index",
    }
xclip = result_map.get("XCLIP", {})
if xclip.get("latency_seconds", {}).get("p50") and average_video_seconds:
    clip_count = math.ceil(average_video_seconds / 4.0) * len(videos)
    EXTRAPOLATIONS["xclip_precompute_if_chosen"] = {
        "assumed_stride_seconds": 4.0,
        "estimated_clip_count": clip_count,
        "estimated_hours": clip_count * xclip["latency_seconds"]["p50"] / 3600,
        "estimated_fp16_512d_storage_bytes": clip_count * 512 * 2,
    }

DEPENDENCY_MATRIX = {
    "strategy": "OPTIONAL_LAZY_IMPORTS_PLUS_PREPROCESSING_ARTIFACTS_AND_ISOLATED_OCR",
    "shared_transformers_environment": ["XCLIP", "WHISPER_TURBO", "QWEN25VL3B_FP16", "SMOLVLM2_500M", "GROUNDING_DINO_TINY", "SAM21_TINY"],
    "isolated_or_dependency_risk": ["PPOCRV5_MOBILE_VI", "QWEN25VL3B_AWQ", "INTERNVIDEO2_DISTILLED"],
    "hard_pyproject_dependencies_to_add": [],
    "flash_attention_2_required": False,
    "heavy_models_simultaneously_loaded": False,
    "commands": COMMAND_LOG,
    "resolved_packages": {
        name: package_version(name)
        for name in ("torch", "torchvision", "transformers", "accelerate", "qwen-vl-utils", "av", "soundfile", "psutil")
    },
}

PRELIM_STACK = {
    "name": "FS1_PRELIM_SINGLE_EXECUTABLE_STACK",
    "hardware": "KAGGLE_T4_16GB",
    "core": ["FROZEN_OPENAI_CLIP", "FROZEN_SIGLIP2_INDEX", "FROZEN_OPUS_MT", "BTC_OBJECT_METADATA", "T3_G1_BCF", "QUERY_LOCAL_EVENT_GRAPH_DATACLASSES", "DETERMINISTIC_DP_BEAM"],
    "video_action": video_recommendation,
    "ocr": "PPOCRV5_PRECOMPUTED_ARTIFACT" if passed("PPOCRV5_MOBILE_VI") else "DISABLED",
    "asr": "WHISPER_TURBO_PRECOMPUTED_ARTIFACT" if passed("WHISPER_TURBO") else "DISABLED",
    "vlm": "QWEN25VL3B_LAZY_QUERY_LOCAL" if passed("QWEN25VL3B_FP16") else ("SMOLVLM2_500M_LAZY_QUERY_LOCAL" if passed("SMOLVLM2_500M") else "DISABLED"),
    "region": "GROUNDING_DINO_TINY_LAZY_QUERY_LOCAL" if passed("GROUNDING_DINO_TINY") else "DISABLED",
    "tracking": "SAM21_TINY_LAZY_SHORT_CLIP" if passed("SAM21_TINY") else "DISABLED",
    "load_policy": "ONE_HEAVY_MODEL_AT_A_TIME_WITH_EXPLICIT_UNLOAD",
    "production_promotion": False,
}
LOCAL_STACK = {
    "name": "FS1_FINALS_LOCAL_4GB_ESTIMATED_PROFILE",
    "hardware": "RTX3050TI_LAPTOP_4GB_ESTIMATED_ONLY",
    "verification": "ESTIMATED_ONLY_NOT_RUN_ON_TARGET_HARDWARE",
    "core": ["FROZEN_RETRIEVAL_CPU_OR_SINGLE_SMALL_GPU_COMPONENT", "QUERY_LOCAL_EVENT_GRAPH", "DETERMINISTIC_DP_BEAM"],
    "video_action": "XCLIP_ONLY_IF_MEASURED_PEAK_FITS_OTHERWISE_DISABLED",
    "ocr": "PRECOMPUTED_OR_CPU",
    "asr": "WHISPER_SMALL_PRECOMPUTE_SEPARATE_PROCESS",
    "vlm": "SMOLVLM2_500M_LAZY_SINGLE_FRAME_OR_SHORT_VIDEO",
    "region": "GROUNDING_DINO_TINY_SEQUENTIAL_OR_CPU_OFFLOAD",
    "tracking": "SAM21_TINY_SEQUENTIAL_FP16_IF_TARGET_TEST_PASSES",
    "qwen25vl3b": "NOT_DEFAULT",
}

EVENT_GRAPH = {
    "scope": "QUERY_LOCAL_IN_MEMORY_ONLY",
    "implementation_shape": "Python dataclasses plus adjacency lists",
    "nodes": ["VideoHypothesis", "QueryEvent", "EventCandidate", "EvidenceRef", "SemanticMomentHypothesis", "EntityHypothesis(optional)"],
    "edges": ["SUPPORTS", "CONTRADICTS", "PRECEDES", "OVERLAPS", "ADJACENT", "ANCHORS", "POSSIBLE_SAME_ENTITY"],
    "temporal_solver": "DETERMINISTIC_DP_OR_BEAM",
    "forbidden": ["Neo4j", "GNN", "full-corpus graph database"],
}

print({"video_recommendation": video_recommendation, "vlm_recommendation": vlm_recommendation, "classifications": CLASSIFICATIONS})


In [ ]:
from datetime import UTC, datetime
from zipfile import ZIP_DEFLATED, ZipFile, ZipInfo

def write_json(name, value):
    path = OUTPUT_ROOT / name
    path.write_text(json.dumps(value, indent=2, default=str) + "\n", encoding="utf-8")
    return path

RESULT_DOCUMENT = {
    "experiment": "FS1_ENGINEERING_FEASIBILITY_RESEARCH",
    "created_at": datetime.now(UTC).isoformat(),
    "hardware_profile_a": SYSTEM_PROFILE,
    "hardware_profile_b": "ESTIMATED_ONLY",
    "sample_manifest": SAMPLE_MANIFEST,
    "candidate_results": RESULTS,
    "classifications": CLASSIFICATIONS,
    "video_recommendation": video_recommendation,
    "vlm_recommendation": vlm_recommendation,
    "extrapolations": EXTRAPOLATIONS,
    "event_graph": EVENT_GRAPH,
    "gt_used": False,
    "team_eval_run": False,
    "fs1_implemented": False,
    "production_policy_changed": False,
    "siglip2_rebuilt": False,
}
MODEL_ASSET_PLAN = {
    "policy": "PIN_EXACT_REVISION_AND_MATERIALIZE_ONCE_AS_KAGGLE_DATASET",
    "assets": [
        {key: row.get(key) for key in ("candidate", "model_id", "exact_revision", "license", "asset_size_bytes", "offline_local_files_only_reload", "status")}
        for row in RESULTS if row.get("model_id")
    ],
    "runtime_internet_required": False,
    "research_download_internet_required": True,
    "weights_in_report_bundle": False,
}
write_json("fs1_engineering_feasibility.json", RESULT_DOCUMENT)
write_json("fs1_model_asset_plan.json", MODEL_ASSET_PLAN)
write_json("fs1_dependency_matrix.json", DEPENDENCY_MATRIX)
write_json("fs1_prelim_recommended_stack.json", PRELIM_STACK)
write_json("fs1_local_4gb_recommended_stack.json", LOCAL_STACK)

rows = []
for result in RESULTS:
    rows.append(json.dumps(result, ensure_ascii=False, default=str))
(OUTPUT_ROOT / "fs1_candidate_results.jsonl").write_text("\n".join(rows) + "\n", encoding="utf-8")

lines = [
    "# FS1 Engineering Feasibility Report", "",
    "> Research-only. No FS1 implementation, TEAM-EVAL, GT access, SigLIP2 rebuild, or production change occurred.", "",
    "## Runtime profile", "",
    f"- GPU: `{SYSTEM_PROFILE['gpu']['name']}`",
    f"- Compute capability: `{SYSTEM_PROFILE['gpu']['compute_capability']}`",
    f"- VRAM: `{SYSTEM_PROFILE['gpu']['total_vram_bytes']}` bytes",
    f"- Python: `{sys.version.split()[0]}`",
    f"- PyTorch/CUDA: `{torch.__version__}` / `{torch.version.cuda}`",
    f"- Transformers: `{package_version('transformers')}`",
    f"- BF16 supported reported by PyTorch: `{SYSTEM_PROFILE['gpu']['bf16_supported_by_torch']}`; FP16 was used.",
    "- FlashAttention-2: not installed or required by this protocol.", "",
    "## Candidate measurements", "",
    "| Candidate | Status | Revision | Asset GiB | Peak reserved GiB | p50 s | Offline reload | RTX3050Ti 4GB |",
    "|---|---|---|---:|---:|---:|---|---|",
]
for row in RESULTS:
    latency = row.get("latency_seconds", {}).get("p50")
    lines.append(
        "| {candidate} | {status} | {revision} | {asset} | {peak} | {latency} | {offline} | {local} |".format(
            candidate=row.get("candidate"), status=row.get("status"),
            revision=(row.get("exact_revision") or "-")[:12],
            asset=round(row.get("asset_size_bytes", 0) / 1024**3, 3) if row.get("asset_size_bytes") else "-",
            peak=round(row.get("gpu_peak_reserved_bytes", 0) / 1024**3, 3) if row.get("gpu_peak_reserved_bytes") else "-",
            latency=round(latency, 4) if latency is not None else "-",
            offline=row.get("offline_local_files_only_reload", "-"),
            local=row.get("rtx3050ti_4gb_status", "UNKNOWN"),
        )
    )
lines.extend([
    "", "## Direct answers", "",
    f"- Video/action recommendation: **{video_recommendation}**.",
    "- InternVideo2 distilled: use only if discovery produced a non-gated official Transformers S14/B14 and a later isolated T4 inference test passes; otherwise DEFER.",
    f"- VLM recommendation: **{vlm_recommendation}**.",
    "- OCR and ASR are preprocessing-only; their heavy runtimes must not remain loaded during query inference.",
    "- Qwen uses FP16 plus SDPA without FlashAttention-2 and bounded pixels.",
    "- AWQ remains optional and is marked dependency risk whenever its worker fails or needs extra kernels.",
    "- Grounding DINO and SAM2 are query-local lazy plugins only.",
    "- RTX 3050 Ti conclusions are ESTIMATED_ONLY until run on that GPU.", "",
    "## Dependency strategy", "",
    "Use optional lazy imports in the query runtime; use reusable artifacts for OCR/ASR; isolate PaddleOCR, AWQ, and any InternVideo source runtime. Add no heavy hard dependency to the TRIAGE-EG package.", "",
    "## Event Graph engineering shape", "",
    "Query-local Python dataclasses and adjacency lists organize evidence and missing-event state. Deterministic DP/beam remains the temporal solver. No Neo4j, GNN, or corpus graph database.", "",
    "## Preliminary stack", "", "```json", json.dumps(PRELIM_STACK, indent=2), "```", "",
    "## Local 4 GB profile", "", "```json", json.dumps(LOCAL_STACK, indent=2), "```", "",
    "## Extrapolations", "", "All estimates below are explicitly derived from the bounded sample and are not full-corpus measurements.", "", "```json", json.dumps(EXTRAPOLATIONS, indent=2), "```", "",
    "## Exact commands", "", "```json", json.dumps(COMMAND_LOG, indent=2, default=str), "```", "",
    "## Blockers and failures", "",
])
for row in RESULTS:
    if row.get("status") not in {"PASS", "PASS_WITH_LIMITS"}:
        lines.append(f"- `{row.get('candidate')}`: `{row.get('status')}` — {json.dumps(row.get('errors') or row.get('blocker') or row.get('reason'), default=str)}")
lines.extend(["", "## Stop", "", "Research artifacts complete. FS1 was not implemented.", ""])
(OUTPUT_ROOT / "FS1_ENGINEERING_FEASIBILITY_REPORT.md").write_text("\n".join(lines), encoding="utf-8")

files = sorted(path for path in OUTPUT_ROOT.rglob("*") if path.is_file())
with ZipFile(ZIP_PATH, "w", ZIP_DEFLATED, allowZip64=True) as archive:
    for path in files:
        info = ZipInfo(path.relative_to(OUTPUT_ROOT).as_posix(), (1980, 1, 1, 0, 0, 0))
        info.compress_type = ZIP_DEFLATED
        info.external_attr = 0o644 << 16
        archive.writestr(info, path.read_bytes())
print({
    "status": "COMPLETE",
    "candidate_count": len(RESULTS),
    "output_files": [path.name for path in files],
    "download_zip": str(ZIP_PATH),
    "zip_size_bytes": ZIP_PATH.stat().st_size,
    "fs1_implemented": False,
    "production_policy_changed": False,
    "STOP": True,
})
